In [1]:
import pandas as pd
import numpy as np

In [2]:
PATH = 'data/ContingencyDataASARCO.csv'
df_complete = pd.read_csv(PATH, parse_dates=['timestamp'])

In [112]:
df_complete.head()

,id,fieldid,shiftid,timestamp,duration,truck_id,stop_code,stop_reason,status_id,status_name,timecat,timecatid
0,2403070010020393380,04:40:28,240307001,2024-03-07 01:40:28,1417,CA84,314,SIN OPERADOR / FUERA DE FAENA,225,Reserva,Reserva,232
1,2403070010022362204,05:09:18,240307001,2024-03-07 02:09:18,2000,CA105,314,SIN OPERADOR / FUERA DE FAENA,225,Reserva,Reserva,232
2,2403070010025988300,06:19:37,240307001,2024-03-07 03:19:37,938,CA80,314,SIN OPERADOR / FUERA DE FAENA,225,Reserva,Reserva,232
3,2403070010025988552,06:19:44,240307001,2024-03-07 03:19:44,4271,CA62,314,SIN OPERADOR / FUERA DE FAENA,225,Reserva,Reserva,232
4,2403070020000036708,00:00:00,240307002,2024-03-07 09:00:00,43200,CA108,411,Falla motor diesel,223,Mantencion,Mant Nprog,235


In [122]:
group_asarco = ['stop_reason', 'timecat']
additional_cols = ['duration']
resume_activities = df_complete[[*group_asarco, *additional_cols]].groupby(by=group_asarco).\
                                                                   agg({'timecat': 'count', 'duration': 'sum'}).\
                                                                   rename(columns={'timecat': 'count', 'duration': 'total_duration'}).\
                                                                   reset_index().\
                                                                   sort_values('count', ascending=False)

In [116]:
resume_activities.groupby(by=['timecat'])['timecat'].count().rename('count')

timecat
Accidente        2
Demora Nprog    13
Demora Prog      5
Efectivo         2
MARC             9
Mant Nprog      11
Mant Prog        9
Reserva          9
Name: count, dtype: int64

In [117]:
# MARC: Maintenance and Repair Contract
resume_activities[resume_activities['timecat'] == 'MARC']

,stop_reason,timecat,count,duration
54,Sin infraestructura de apoyo,MARC,103,3349829
23,INSTALACIÓN SISTEMA TECNOLÓGI,MARC,22,361621
51,SISTEMA MESH COASIN,MARC,19,76074
52,SMARTCAP,MARC,9,25858
55,Sistema contra incendios,MARC,7,165966
42,Radiocomunicacion,MARC,6,10802
50,SISTEMA CAS,MARC,6,82426
56,Sistema dispatch - modular,MARC,2,5063
11,Cambio vidrios,MARC,1,8835


In [6]:
import plotly.express as px

In [110]:
df_translation = pd.read_csv('data/mapped_stop_reasons.csv')

In [118]:
resume_activities_asarco = resume_activities.copy()
asarco_activities = resume_activities['timecat'].unique()
asarco_values = ['Efectivo', 'Demora No Prog', 'Demora Prog', 'Mant No Prog', 'Reserva', 'Mant Prog', 'MARC', 'Accidente']
asarco_mapper = dict(zip(asarco_activities, asarco_values))
resume_activities_asarco['timecat'] = resume_activities_asarco['timecat'].map(asarco_mapper)
resume_activities_asarco = pd.merge(df_translation[['stop_reason', 'manual_parsing']],\
                                    resume_activities_asarco,\
                                    left_on='stop_reason',\
                                    right_on='stop_reason')
resume_activities_asarco = resume_activities_asarco.drop(columns='stop_reason').\
                                                    rename(columns={'manual_parsing': 'event', 'timecat': 'asarco_event'})

In [12]:
df_sunburst = resume_activities_asarco.reset_index(drop=True)
df_sunburst['values'] = 1
df_sun = pd.merge(df_translation, df_sunburst, left_on='manual_parsing', right_on='event')
df_sun = df_sun.drop(columns=['values', 'event'])

TO_SAVE = False
if TO_SAVE:
  SAVE_PATH = 'data/mapped_asarco_stop_reasons.csv'
  df_sun.to_csv(SAVE_PATH, index=False)

In [19]:
# resume_activities_asarco.columns has ['event', 'asarco_event', 'count']
df_tree = resume_activities_asarco.copy()
df_tree['parent'] = 'Clasificación ASARCO'
df_tree['value'] = 1
df_tree['truncated_event'] = df_tree['event'].str[0:10] + '...'
color_palette = ['#E6BA95', '#FAFDD6', '#E4E9BE', '#A2B38B', '#B3E2A7', '#98ABEE', '#FFF3CF', '#FFC7EA']

fig = px.treemap(df_tree, 
                 path=['parent', 'asarco_event', 'truncated_event'],
                 values='value',
                 color_discrete_sequence=color_palette)
fig.update_layout(
  font=dict(
    family='Arial',
    size=18,
    color='black'
  ),
  uniformtext=dict(
    minsize=8
  )
)

FOR_SAVE = False
if FOR_SAVE:
  SAVE_PATH = 'images/ASARCOTreeMapUnique.png'
  SAVE_SPECS = {'width': 1500, 'height': 700, 'scale': 1}
  fig.write_image(SAVE_PATH, **SAVE_SPECS)

fig.show()

In [20]:
# Distribución por Frecuencia total
df_tree = resume_activities_asarco.copy()
df_tree['parent'] = 'Clasificación ASARCO (Frecuencia absoluta)'
df_tree['value'] = 1
df_tree['truncated_event'] = np.where(df_tree['event'].str.len() > 8, df_tree['event'].str[0:8] + '...', df_tree['event'])
color_palette = ['#E6BA95', '#FAFDD6', '#E4E9BE', '#A2B38B', '#B3E2A7', '#98ABEE', '#FFF3CF', '#FFC7EA']

fig = px.treemap(df_tree,
                 path=['parent', 'asarco_event', 'truncated_event'],
                 values='count',
                 color_discrete_sequence=color_palette)

fig.update_layout(
  font=dict(
    family='Arial',
    size=18,
    color='black'
  ),
  uniformtext=dict(
    minsize=8,
    # mode='' # Can be 'hide' to hide if the text is too large
  )
)

FOR_SAVE = False
if FOR_SAVE:
  SAVE_PATH = 'images/ASARCOTreeMapUniqueByFreq.png'
  SAVE_SPECS = {'width': 1500, 'height': 700, 'scale': 1}
  fig.write_image(SAVE_PATH, **SAVE_SPECS)

fig.show()

In [21]:
# Distribución por Frecuencia total
df_tree = resume_activities_asarco.copy()
df_tree = df_tree[df_tree['asarco_event'] != 'Efectivo']
df_tree['parent'] = 'Clasificación ASARCO sin Efectivo (Frecuencia absoluta)'
df_tree['value'] = 1
df_tree['truncated_event'] = np.where(df_tree['event'].str.len() > 8, df_tree['event'].str[0:8] + '...', df_tree['event'])
color_palette = ['#E6BA95', '#FAFDD6', '#E4E9BE', '#A2B38B', '#B3E2A7', '#98ABEE', '#FFF3CF', '#FFC7EA']

fig = px.treemap(df_tree,
                 path=['parent', 'asarco_event', 'truncated_event'],
                 values='count',
                 color_discrete_sequence=color_palette)

fig.update_layout(
  font=dict(
    family='Arial',  # Specify the font family
    size=18,         # Set the font size
    color='black'    # Set the font color
  ),
  uniformtext=dict(
    minsize=8,
  )
)

FOR_SAVE = True
if FOR_SAVE:
  SAVE_PATH = 'images/ASARCOTreeMapByFreqWithoutEffective.png'
  SAVE_SPECS = {'width': 1500, 'height': 700, 'scale': 1}
  fig.write_image(SAVE_PATH, **SAVE_SPECS)

fig.show()

In [137]:
summary_asarco = df_tree.groupby(by='asarco_event')['count'].sum().to_frame().reset_index()
summary_asarco['relative_perc'] = ((summary_asarco['count'] / summary_asarco['count'].sum()) * 100).round(2)
summary_asarco = summary_asarco.sort_values(by='relative_perc', ascending=False)
summary_asarco['cumulative'] = summary_asarco['relative_perc'].cumsum()
summary_asarco

,asarco_event,count,relative_perc,cumulative
1,Demora No Prog,30171,74.69,74.69
2,Demora Prog,7619,18.86,93.55
6,Reserva,1245,3.08,96.63
4,Mant No Prog,802,1.99,98.62
5,Mant Prog,362,0.90,99.52
3,MARC,175,0.43,99.95
0,Accidente,20,0.05,100.00


In [185]:
from plot_utils import plot_pareto
from plotly.graph_objects import Figure

def save_figure(fig: Figure, file_name: str, folder: str = 'images') -> None:
  SAVE_IMAGE = True
  if SAVE_IMAGE:
    SAVE_PATH = f'{folder}/{file_name}.png'
    SAVE_SPECS = {'width': 1500, 'height': 550, 'scale': 1}
    fig.write_image(SAVE_PATH, **SAVE_SPECS)

In [139]:
summary_asarco = df_complete.groupby(by='timecat').agg({'duration': ['count', 'sum', 'mean', 'median']})
summary_asarco.columns = summary_asarco.columns.get_level_values(1)

In [140]:
summary_asarco = summary_asarco.reset_index()
summary_asarco['lost_time_hours'] = summary_asarco['sum'] / (60 * 60) # Timeloss in hours
summary_asarco['lost_time_days'] = summary_asarco['sum'] / (60 * 60 * 24) # Timeloss in days
summary_asarco = summary_asarco[summary_asarco['timecat'] != 'Efectivo']
summary_asarco

,timecat,count,sum,mean,median,lost_time_hours,lost_time_days
0,Accidente,20,551252,27562.600000,34083.0,153.125556,6.380231
1,Demora Nprog,30171,9187972,304.529913,165.0,2552.214444,106.342269
2,Demora Prog,7619,10132808,1329.939362,1080.0,2814.668889,117.277870
4,MARC,175,4086474,23351.280000,19522.0,1135.131667,47.297153
5,Mant Nprog,802,11597449,14460.659601,10021.0,3221.513611,134.229734
6,Mant Prog,362,6324897,17472.091160,12727.0,1756.915833,73.204826
7,Reserva,1245,2871972,2306.804819,1205.0,797.770000,33.240417


In [186]:
fig = plot_pareto(summary_asarco, 'timecat', 'count', 'Pareto frecuencia de actividades')
save_figure(fig, 'pareto_freq')
fig

In [187]:
fig = plot_pareto(summary_asarco, 'timecat', 'lost_time_days', 'Pareto ASARCO tiempos totales en días', 'Tiempo total en días')
save_figure(fig, 'pareto_tiempo_total_asarco')
fig

In [188]:
# Pareto for top
resume_activities_asarco['lost_time_days'] = resume_activities_asarco['duration'] / (24 * 60 * 60)
df_mant_prog = resume_activities_asarco[resume_activities_asarco['asarco_event'] == 'Mant No Prog']
fig = plot_pareto(df_mant_prog, 'event', 'lost_time_days', 'Pareto Mantenimiento No programado', 'Tiempo total en días')
save_figure(fig, 'pareto_tiempo_mant_no_programado')
fig

In [244]:
df_demora_prog = resume_activities_asarco[resume_activities_asarco['asarco_event'] == 'Demora Prog']
fig = plot_pareto(df_demora_prog, 'event', 'lost_time_days', 'Pareto en Demoras Programadas', 'Tiempo total en días')
save_figure(fig, 'pareto_tiempo_demoras_programadas')
fig

In [192]:
df_demora_prog = resume_activities_asarco[resume_activities_asarco['asarco_event'] == 'Demora No Prog']
fig = plot_pareto(df_demora_prog, 'event', 'lost_time_days', 'Pareto en Demoras no Programadas', 'Tiempo total en días')
save_figure(fig, 'pareto_tiempo_demoras_no_programadas')
fig

In [194]:
df_mant_prog = resume_activities_asarco[resume_activities_asarco['asarco_event'] == 'Mant Prog']
fig = plot_pareto(df_mant_prog, 'event', 'lost_time_days', 'Pareto en Mantencion Programada', 'Tiempo total en días')
save_figure(fig, 'pareto_tiempo_mantenimiento_programado')
fig

In [218]:
resume_activities_2 = resume_activities_asarco.copy()
resume_activities_2['lost_time_days'] = resume_activities_2['duration'] / (60 * 60 * 24)
resume_activities_2 = resume_activities_2.sort_values('duration', ascending=False)
group = resume_activities_2.groupby(by='asarco_event')
resume_activities_2['cumulate'] = (resume_activities_2['lost_time_days'] / group['lost_time_days'].transform('sum')) * 100
resume_activities_2['cumulate'] = resume_activities_2.groupby(by='asarco_event')['cumulate'].cumsum()

In [231]:
def catch_exceeding_and_previous(group):
  idx = group[group['cumulate'] > 80].index[0]
  return group.loc[:idx]

result = group.apply(catch_exceeding_and_previous)

/tmp/ipykernel_20895/1165594714.py:5: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [243]:
result.rename(columns={'asarco_event': 'asarco_event2'}).\
       reset_index(level=[0,1]).\
       drop(columns=['asarco_event2', 'level_1', 'count', 'duration', 'cumulate']).\
       query('asarco_event == ("Demora No Prog", "Demora Prog", "Mant No Prog", "Mant Prog")')

,asarco_event,event,lost_time_days
1,Demora No Prog,Pista obstruida,34.105104
2,Demora No Prog,Sin equipo de carguio,24.019155
3,Demora No Prog,Operador fuera del equipo,20.872650
4,Demora No Prog,Carga de combustible,11.279653
5,Demora Prog,Cambio de turno,35.101806
6,Demora Prog,Colacion en comedor,34.981725
7,Demora Prog,Colacion en cabina 2,29.258866
10,Mant No Prog,Mecanica imprevista,75.125810
11,Mant No Prog,Falla motor diesel,48.534421
12,Mant Prog,Mantencion programada,57.373171
